# JAXTPC Simulation

GPU-accelerated liquid argon TPC detector simulation.

**Workflow:**
1. Load detector configuration and particle segment data
2. Run simulation (recombination, drift, diffusion, wire response)
3. Convert output to dense or sparse format
4. Visualize: detector response, truth hits, track labels

**Settings:** Toggle flags in the configuration cell.

## Setup and Configuration

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

CONFIG_PATH = "config/cubic_wireplane_config.yaml"
DATA_PATH = "out_500.h5"
EVENT_IDX = 7

# Accumulation mode
USE_BUCKETED = True
MAX_ACTIVE_BUCKETS = 1000

# Post-processing inside JIT
INCLUDE_NOISE = False
INCLUDE_ELECTRONICS = False
INCLUDE_DIGITIZATION = True
INCLUDE_ELECTRIC_DISTORTIONS = False

# Track labeling
INCLUDE_TRACK_HITS = True

# Threshold for sparse visualization (electrons)
THRESHOLD_ENC = 1200

# Performance knobs
TOTAL_PAD = 200_000          # Padded array size per side (sets JIT shape)
RESPONSE_CHUNK_SIZE = 50_000 # Deposits per fori_loop iteration
MAX_KEYS = 2_000_000         # Max unique hits for track labeling (only when enabled)

print("Configuration:")
print(f"  Config: {CONFIG_PATH}")
print(f"  Data: {DATA_PATH}, Event: {EVENT_IDX}")
print(f"  Bucketed: {USE_BUCKETED}")
print(f"  Noise: {'ON' if INCLUDE_NOISE else 'OFF'}")
print(f"  Electronics: {'ON' if INCLUDE_ELECTRONICS else 'OFF'}")
print(f"  Digitization: {'ON' if INCLUDE_DIGITIZATION else 'OFF'}")
print(f"  Track labeling: {'ON' if INCLUDE_TRACK_HITS else 'OFF'}")
print(f"  Threshold: {THRESHOLD_ENC} e-")
print(f"  total_pad: {TOTAL_PAD:,}, chunk: {RESPONSE_CHUNK_SIZE:,}, max_keys: {MAX_KEYS:,}")

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import os
import time
import gc

from tools.simulation import DetectorSimulator
from tools.geometry import generate_detector
from tools.loader import load_event
from tools.config import create_track_hits_config
from tools.output import to_dense, to_sparse

from tools.visualization import (
    visualize_wire_signals,
    visualize_diffused_charge,
    visualize_track_labels,
    get_top_tracks_by_charge,
)

os.makedirs("plots", exist_ok=True)

print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

## Load Data

In [ ]:
# =============================================================================
# LOAD DATA
# =============================================================================

detector_config = generate_detector(CONFIG_PATH)

# Create simulator first (needed for config to build deposits)
jax.clear_caches()
gc.collect()

track_config = create_track_hits_config(max_keys=MAX_KEYS) if INCLUDE_TRACK_HITS else None

simulator = DetectorSimulator(
    detector_config,
    use_bucketed=USE_BUCKETED,
    max_active_buckets=MAX_ACTIVE_BUCKETS,
    include_noise=INCLUDE_NOISE,
    include_electronics=INCLUDE_ELECTRONICS,
    include_track_hits=INCLUDE_TRACK_HITS,
    include_electric_dist=INCLUDE_ELECTRIC_DISTORTIONS,
    include_digitize=INCLUDE_DIGITIZATION,
    total_pad=TOTAL_PAD,
    response_chunk_size=RESPONSE_CHUNK_SIZE,
    track_config=track_config,
)

cfg = simulator.config

print(f"\nLoading event {EVENT_IDX} from {DATA_PATH}...")
deposits = load_event(DATA_PATH, cfg, event_idx=EVENT_IDX)

n_volumes = len(deposits.volumes)
n_total = sum(v.n_actual for v in deposits.volumes)
print(f"Loaded {n_total:,} deposits across {n_volumes} volumes")
for v in range(n_volumes):
    vol = deposits.volumes[v]
    print(f"  Volume {v}: {vol.n_actual:,} deposits")

## Create Simulator and Run

In [ ]:
# =============================================================================
# WARM UP + RUN
# =============================================================================

print("Warming up JIT...")
simulator.warm_up()
print("Done.")

In [ ]:
# =============================================================================
# RUN SIMULATION
# =============================================================================

print("Running simulation...")
t0 = time.time()

response_signals, track_hits_raw, deposits = simulator.process_event(
    deposits, key=jax.random.PRNGKey(42))

# Wait for GPU
for val in response_signals.values():
    if isinstance(val, tuple):
        jax.block_until_ready(val[0])
    else:
        jax.block_until_ready(val)

elapsed = time.time() - t0
print(f"Simulation: {elapsed:.2f}s ({n_total / elapsed:,.0f} segments/sec)")
print(f"Output format: {cfg.output_format}")

## Convert Output

Convert simulation output (dense, bucketed, or wire-sparse) to sparse format:
- **Sparse**: `{(vol, plane): {'wire': (N,), 'time': (N,), 'values': (N,)}}` — for scatter plots and storage

In [ ]:
# =============================================================================
# CONVERT TO SPARSE
# =============================================================================

threshold_adc = THRESHOLD_ENC / cfg.electrons_per_adc
t0 = time.time()
sparse_signals = to_sparse(response_signals, cfg, threshold_adc=threshold_adc)
print(f"to_sparse ({THRESHOLD_ENC} e-): {time.time() - t0:.2f}s")

# Statistics
print(f"\nSparse output (threshold={THRESHOLD_ENC} e-):")
print(f"{'Plane':<16} {'Pixels':>10} {'Sparsity':>10}")
print("-" * 40)
for (vi, pi), data in sorted(sparse_signals.items()):
    n_pix = len(data['values'])
    total = cfg.volumes[vi].num_wires[pi] * cfg.num_time_steps
    sparsity = (1 - n_pix / total) * 100
    ptype = cfg.plane_names[vi][pi]
    print(f"  Vol {vi} {ptype:<5} {n_pix:>10,} {sparsity:>9.1f}%")

## Finalize Track Hits

In [ ]:
# =============================================================================
# FINALIZE TRACK HITS
# =============================================================================

if INCLUDE_TRACK_HITS:
    track_hits = simulator.finalize_track_hits(track_hits_raw)

    total_hits = 0
    print("Track hits per plane:")
    print(f"{'Plane':<16} {'Hits':>10} {'Labeled':>10}")
    print("-" * 40)
    for (vi, pi), data in sorted(track_hits.items()):
        nh = int(data['num_hits'])
        nl = int(data['num_labeled'])
        total_hits += nh
        ptype = cfg.plane_names[vi][pi]
        print(f"  Vol {vi} {ptype:<5} {nh:>10,} {nl:>10,}")
    print("-" * 40)
    print(f"  {'Total':<14} {total_hits:>10,}")
else:
    track_hits = None
    print("Track labeling disabled.")

## Visualization

1. **Response signals** (dense): full detector view with imshow
2. **Truth hits**: diffused charge colored by magnitude
3. **Track labels**: pixels colored by dominant track ID

In [ ]:
# =============================================================================
# VISUALIZE RESPONSE SIGNALS (SPARSE)
# =============================================================================

fig = visualize_wire_signals(
    sparse_signals, cfg,
    threshold_enc=THRESHOLD_ENC, gamma=0.2,
    sparse=True,
    )
fig.savefig("plots/response_new.png", dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# =============================================================================
# VISUALIZE TRUTH HITS (SPARSE)
# =============================================================================

if INCLUDE_TRACK_HITS:
    # Build sparse truth data from track hits (no densification needed)
    truth_sparse = {}
    for (vi, pi), data in track_hits.items():
        nh = int(data['num_hits'])
        if nh > 0:
            hbt = np.array(data['hits_by_track'][:nh])
            truth_sparse[(vi, pi)] = {
                'wire': hbt[:, 0].astype(np.int32),
                'time': hbt[:, 1].astype(np.int32),
                'values': hbt[:, 2].astype(np.float32),
            }
        else:
            truth_sparse[(vi, pi)] = {
                'wire': np.array([], dtype=np.int32),
                'time': np.array([], dtype=np.int32),
                'values': np.array([], dtype=np.float32),
            }

    fig_truth = visualize_diffused_charge(truth_sparse, cfg,
                                           log_norm=True, threshold=50,
                                           sparse=True)
    fig_truth.suptitle(f'Event {EVENT_IDX} — Truth Hits ({total_hits:,} hits)',
                       fontsize=14, y=1.02)
    fig_truth.savefig("plots/truth_hits_new.png", dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
else:
    print("Track labeling disabled.")

In [ ]:
# =============================================================================
# VISUALIZE TRACK LABELS
# =============================================================================

if INCLUDE_TRACK_HITS:
    top_tracks = get_top_tracks_by_charge(track_hits, top_n=20)

    # Count unique tracks across all volumes
    all_tids = np.concatenate([
        np.asarray(deposits.volumes[v].track_ids[:deposits.volumes[v].n_actual])
        for v in range(n_volumes)
    ])
    n_unique_tracks = len(np.unique(all_tids))

    if top_tracks:
        print("Top 10 tracks by charge:")
        for i, (tid, charge) in enumerate(top_tracks[:10]):
            print(f"  {i+1:2d}. Track {tid:4d}: {charge:12,.1f}")

    fig_tracks = visualize_track_labels(track_hits, cfg, top_tracks, max_tracks=15)
    fig_tracks.suptitle(f'Event {EVENT_IDX} — Track Labels ({n_unique_tracks:,} tracks)',
                        fontsize=14, y=1.02)
    fig_tracks.savefig("plots/track_labels_new.png", dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
else:
    print("Track labeling disabled.")

In [ ]:
# =============================================================================
# SINGLE TRACK VISUALIZATION (SPARSE)
# =============================================================================

TRACK_RANK = 2  # Nth most active track (1-indexed)

if INCLUDE_TRACK_HITS and top_tracks and TRACK_RANK <= len(top_tracks):
    selected_tid, selected_charge = top_tracks[TRACK_RANK - 1]
    print(f"Rank {TRACK_RANK}: Track {selected_tid}, charge={selected_charge:,.1f}")

    single_sparse = {}
    total_track_hits = 0

    for (vi, pi), data in track_hits.items():
        nl = int(data['num_labeled'])
        if nl > 0:
            labeled = np.array(data['labeled_hits'][:nl])
            tids = np.array(data['labeled_track_ids'][:nl])
            mask = tids == selected_tid

            if mask.any():
                single_sparse[(vi, pi)] = {
                    'wire': labeled[mask, 0].astype(np.int32),
                    'time': labeled[mask, 1].astype(np.int32),
                    'values': labeled[mask, 2].astype(np.float32),
                }
                total_track_hits += int(mask.sum())
            else:
                single_sparse[(vi, pi)] = {
                    'wire': np.array([], dtype=np.int32),
                    'time': np.array([], dtype=np.int32),
                    'values': np.array([], dtype=np.float32),
                }
        else:
            single_sparse[(vi, pi)] = {
                'wire': np.array([], dtype=np.int32),
                'time': np.array([], dtype=np.int32),
                'values': np.array([], dtype=np.float32),
            }

    print(f"Track {selected_tid}: {total_track_hits:,} labeled hits")

    fig_single = visualize_diffused_charge(single_sparse, cfg,
                                            log_norm=True, threshold=50,
                                            sparse=True)
    fig_single.suptitle(
        f'Event {EVENT_IDX} — Rank #{TRACK_RANK} Track {selected_tid} ({total_track_hits:,} hits)',
        fontsize=14, y=1.02)
    fig_single.savefig("plots/single_track_new.png", dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
else:
    print("Track labeling disabled or rank not available.")

## Summary

In [ ]:
# =============================================================================
# SUMMARY
# =============================================================================

print("=" * 60)
print(" SIMULATION SUMMARY")
print("=" * 60)

print(f"\nInput:")
print(f"  Data: {DATA_PATH}, Event: {EVENT_IDX}")
print(f"  Total deposits: {n_total:,}")
for v in range(n_volumes):
    print(f"  Volume {v}: {deposits.volumes[v].n_actual:,}")

print(f"\nSimulation:")
print(f"  Time: {elapsed:.2f}s")
print(f"  Throughput: {n_total / elapsed:,.0f} seg/s")
print(f"  Output format: {cfg.output_format}")

if INCLUDE_TRACK_HITS:
    print(f"\nTrack hits: {total_hits:,}")

total_pix = sum(len(d['values']) for d in sparse_signals.values())
total_dense_pix = sum(cfg.volumes[v].num_wires[p] * cfg.num_time_steps
                      for v in range(cfg.n_volumes)
                      for p in range(cfg.volumes[v].n_planes))
print(f"\nSparse output ({THRESHOLD_ENC} e-):")
print(f"  Pixels: {total_pix:,}")
print(f"  Sparsity: {(1 - total_pix/total_dense_pix)*100:.1f}%")

print(f"\nSettings:")
print(f"  Noise: {'ON' if INCLUDE_NOISE else 'OFF'}")
print(f"  Electronics: {'ON' if INCLUDE_ELECTRONICS else 'OFF'}")
print(f"  Digitization: {'ON' if INCLUDE_DIGITIZATION else 'OFF'}")
print(f"  SCE: {'ON' if INCLUDE_ELECTRIC_DISTORTIONS else 'OFF'}")
print(f"  Track labeling: {'ON' if INCLUDE_TRACK_HITS else 'OFF'}")